### Calculate t cell emissions 
- Binary: is contacting another T cell
- Binary: is contacting another cancer cell
- Continuous: Instantaneous velocity
- If overlap morphologies look good
    - Area
    - Circularity

- Optional
    - Binary: is contacting a cancer cell that divides at some point in its life

- Backup features
    - Backup: Cancer cell division within 20um radius
    - Number of T cell neighbors within 20um radius
    - Number of cancer cell neighbors within 20um radius

In [50]:
import os
os.chdir("/gladstone/engelhardt/lab/jutran/lci/MarsonImagingPipeline")
# os.chdir("/gladstone/engelhardt/lab/jadjasu/LiveCellUmbrella/MarsonImagingPipeline")

import pickle
import argparse
from pathlib import Path
import zipfile
from io import BytesIO

import yaml
import tifffile
import numpy as np
from scripts.utils.StatUtils import compute_cell_counts_per_frame, compute_mean_velocity_per_frame, calculate_area, compute_mean_cell_stat_per_frame, count_divisions_per_frame, compute_cell_velocities_per_frame_dict, compute_cell_cell_contact_dict, compute_all_cell_cell_contact_dict
from scripts.utils.CellTypingUtils import filter_tracks
from scripts.utils.PlottingUtils import plot_over_time_by_condition, plot_covariate_scatter_by_condition, plot_mean_covariate_scatter_by_condition
from scripts.utils.StatUtils import *
from scripts.utils.PlottingUtils import *

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np

from scipy import stats

### Load tracks

In [10]:
config_path = Path("/gladstone/engelhardt/lab/jutran/lci/MarsonImagingPipeline/snakemake_configs/Analysis_85t_600xy.yml")
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

cvat_base_dir = "/gladstone/engelhardt/lab/jutran/lci/MarsonImagingPipeline/data/ground_truth_tracking_annotations/cvat_annotations/TCR-T/"
caliban_base_dir = '/gladstone/engelhardt/lab/MarsonLabIncucyteData/UltrackAnalysis/ground_truth_caliban_nuclei_masks/segmentationMasks/'

crop_ids = ['B4_t50t100y200y350x750x900',
            'B8_t50t100y200y350x750x900',
            'E4_t50t100y200y350x750x900',
            'B4_t250t300y200y350x750x900',
            'B8_t250t300y200y350x750x900',
            'E4_t250t300y200y350x750x900']

conditions_dict = {}

conditions_dict["SH"] = ['B4_t50t100y200y350x750x900', 'B4_t250t300y200y350x750x900']
conditions_dict["RASA2"] = ['E4_t50t100y200y350x750x900', 'E4_t250t300y200y350x750x900']
conditions_dict["CUL5"] = ['B8_t50t100y200y350x750x900', 'B8_t250t300y200y350x750x900']

In [11]:
def get_gt_info_per_well(config):

    if not config.get("CONDITIONS"):
        raise ValueError("No conditions found in config file.")
    
    conditions = conditions_dict

    cvat_tracks_per_well = {}
    nuclear_tracks_per_well = {}
    cell_type_dict_per_well = {}

    for condition in conditions:
        for crop in conditions[condition]:
            well_id = crop.split('_')[0]
            cvat_tracks_per_well[crop] = tifffile.imread(os.path.join(cvat_base_dir, well_id, crop, 'ALL_tracks.tiff'))

            # get nuclear tracks
            caliban_tracks = tifffile.imread(os.path.join(caliban_base_dir, f'{crop}.tiff'))
            caliban_tracks = caliban_tracks[...,0]
            nuclear_tracks_per_well[crop] = caliban_tracks
            
            # get cell type dict
            cell_type_dict_per_well[crop] = pickle.load(open(os.path.join(cvat_base_dir, well_id, crop, f'full_cell_type_dict.pkl'), "rb"))
    
    return cvat_tracks_per_well, nuclear_tracks_per_well, cell_type_dict_per_well


In [12]:
# get relevant information
cvat_tracks_per_well, nuclear_tracks_per_well, cell_type_dict_per_well = get_gt_info_per_well(config)

### Calculate relevant statistics

In [13]:
# get tracks by type for each well
type_tracks_per_well = {}
for cell_type in ['cancer', 't_cell', 'nuclei']:
    type_tracks_per_well[cell_type] = {}

    if cell_type == 'nuclei':
        type_tracks_per_well[cell_type] = nuclear_tracks_per_well.copy()
    else:
        type_tracks_per_well[cell_type] = {well: filter_tracks(cell_type, cvat_tracks_per_well[well], cell_type_dict_per_well[well]) for well in cvat_tracks_per_well.keys()}

In [28]:
# calculate t cell velocities
t_cell_velocities_per_frame = {well: compute_cell_velocities_per_frame_dict(t_cell_tracks[well], unit_per_frame=config.get("TIME_FACTOR", 1)) for well in t_cell_tracks.keys()}


Computing cell velocities: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 49/49 [00:02<00:00, 18.29it/s]


In [49]:
# calculate type-specific interactions

cancer_type_specific_contacts_per_frame, t_cell_type_specific_contacts_per_frame = {}, {}
cancer_type_specific_neighbors_per_frame, t_cell_type_specific_neighbors_per_frame = {}, {}

for well in cvat_tracks_per_well.keys():
    t_cell_tracks = type_tracks_per_well["t_cell"][well]
    cancer_tracks = type_tracks_per_well["cancer"][well]

    well_cancer_contacts_per_frame, well_t_cell_contacts_per_frame = compute_cell_cell_contact_dict(t_cell_tracks, cancer_tracks)
    well_cancer_neighbors_per_frame, well_t_cell_neighbors_per_frame = compute_cell_cell_neighbor_dict(t_cell_tracks, cancer_tracks)

    cancer_type_specific_contacts_per_frame[well] = well_cancer_contacts_per_frame
    t_cell_type_specific_contacts_per_frame[well] = well_t_cell_contacts_per_frame

    cancer_type_specific_neighbors_per_frame[well] = well_cancer_neighbors_per_frame
    t_cell_type_specific_neighbors_per_frame[well] = well_t_cell_neighbors_per_frame

Computing cell-cell contact dataframe...


Processing frame-by-frame contact output: 100%|███████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:01<00:00, 25.76it/s]


Computing cell-cell neighbor dataframe...


Processing frame-by-frame neighbor output: 100%|██████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 54.91it/s]


Computing cell-cell contact dataframe...


...

Processing frame-by-frame neighbor output: 100%|██████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 91.84it/s]


Computing cell-cell contact dataframe...


Processing frame-by-frame contact output: 100%|███████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 58.45it/s]


Computing cell-cell neighbor dataframe...


Processing frame-by-frame neighbor output: 100%|██████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 78.73it/s]


In [51]:
# calculate type-agnostic interactions

cancer_all_contacts_per_frame, t_cell_all_contacts_per_frame = {}, {}
cancer_all_neighbors_per_frame, t_cell_all_neighbors_per_frame = {}, {}

for well in cvat_tracks_per_well.keys():
    t_cell_tracks = type_tracks_per_well["t_cell"][well]
    cancer_tracks = type_tracks_per_well["cancer"][well]

    well_cancer_contacts_per_frame, well_t_cell_contacts_per_frame = compute_all_cell_cell_contact_dict(t_cell_tracks, cancer_tracks)
    well_cancer_neighbors_per_frame, well_t_cell_neighbors_per_frame = compute_all_cell_cell_neighbor_dict(t_cell_tracks, cancer_tracks)

    cancer_all_contacts_per_frame[well] = well_cancer_contacts_per_frame
    t_cell_all_contacts_per_frame[well] = well_t_cell_contacts_per_frame

    cancer_all_neighbors_per_frame[well] = well_cancer_neighbors_per_frame
    t_cell_all_neighbors_per_frame[well] = well_t_cell_neighbors_per_frame

Computing cell-cell contact dataframe...


Processing contact dict: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:01<00:00, 32.30it/s]


Computing cell-cell neighbor dataframe...


Processing neighbor dict: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:01<00:00, 40.52it/s]


Computing cell-cell contact dataframe...


...

Processing neighbor dict: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:01<00:00, 32.26it/s]


Computing cell-cell contact dataframe...


Processing contact dict: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:02<00:00, 20.37it/s]


Computing cell-cell neighbor dataframe...


Processing neighbor dict: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:02<00:00, 24.19it/s]


### Create emissions array

In [53]:
example_well_id = 'B4_t50t100y200y350x750x900'

In [59]:
example_cvat_tracks = cvat_tracks_per_well[example_well_id]
example_t_cell_tracks = type_tracks_per_well["t_cell"][example_well_id]
example_cell_type_dict = cell_type_dict_per_well[example_well_id]

example_t_cell_velocities_per_frame = t_cell_velocities_per_frame[example_well_id]
example_t_cell_type_specific_neighbors_per_frame = t_cell_type_specific_neighbors_per_frame[example_well_id]
example_t_cell_all_neighbors_per_frame = t_cell_all_neighbors_per_frame[example_well_id]

In [64]:
t_cell_ids = np.unique(example_t_cell_tracks[example_t_cell_tracks > 0])
id_to_column_index = {cell_id: index for index, cell_id in enumerate(t_cell_ids)}

In [65]:
emissions_array = np.zeros((50, len(t_cell_ids), 3)) # shape: (num_frames, num_t_cells, num_emission_features)

In [70]:
for frame, velocities in example_t_cell_velocities_per_frame.items():
    for cell_id, velocity in velocities.items():
        if cell_id in id_to_column_index:
            column_index = id_to_column_index[cell_id]
            emissions_array[frame, column_index, 0] = velocity

In [74]:
example_t_cell_type_specific_neighbors_per_frame

{0: {np.uint16(3): [1],
  np.uint16(11): [1],
  np.uint16(12): [1],
  np.uint16(13): [1],
  np.uint16(14): [1],
  np.uint16(15): [1],
  np.uint16(16): [1],
  np.uint16(19): [2],
  np.uint16(20): [1],
  np.uint16(21): [3],
  np.uint16(22): [1],
  np.uint16(25): [1],
  np.uint16(30): [1],
  np.uint16(31): [2],
  np.uint16(38): [2],
  np.uint16(39): [1],
  np.uint16(45): [2],
  np.uint16(46): [1],
  np.uint16(48): [1],
  np.uint16(55): [2],
  np.uint16(56): [2],
  np.uint16(95): [2]},
 1: {np.uint16(3): [1],
  np.uint16(11): [1],
  np.uint16(12): [1],
  np.uint16(13): [1],
  np.uint16(14): [1],
  np.uint16(15): [1],
  np.uint16(16): [1],
  np.uint16(19): [2],
  np.uint16(20): [1],
  np.uint16(22): [1],
  np.uint16(25): [1],
  np.uint16(30): [1],
  np.uint16(31): [2],
  np.uint16(38): [2],
  np.uint16(39): [1],
  np.uint16(45): [1],
  np.uint16(46): [1],
  np.uint16(48): [1],
  np.uint16(55): [2],
  np.uint16(56): [2],
  np.uint16(95): [1],
  np.uint16(96): [2]},
 2: {np.uint16(3): [1],
  

In [77]:
for frame in example_t_cell_all_neighbors_per_frame.keys():
    all_neighbors = example_t_cell_all_neighbors_per_frame[frame]
    type_specific_neighbors = example_t_cell_type_specific_neighbors_per_frame[frame]

    for cell_id, neighbors in all_neighbors.items():
        #print(f"frame: {frame}, cell_id: {cell_id}, neighbors: {neighbors}")
        if cell_id in id_to_column_index:
            column_index = id_to_column_index[cell_id]

            cancer_neighbors_list = type_specific_neighbors.get(cell_id, [0])
            cancer_neighbors = cancer_neighbors_list[0]

            t_cell_neighbors = neighbors - cancer_neighbors

            #print(f"cancer_neighbors: {cancer_neighbors}, t_cell_neighbors: {t_cell_neighbors}")

            emissions_array[frame, column_index, 1] = cancer_neighbors
            emissions_array[frame, column_index, 2] = t_cell_neighbors

In [79]:
emissions_array.shape

(50, 110, 3)

In [80]:
np.save('/gladstone/engelhardt/lab/jutran/lci/treeHMM/notebooks/data/example_t_cell_emissions_array.npy', emissions_array)

### Visualize emissions

In [86]:
from sklearn.cluster import KMeans

ImportError: cannot import name 'PCA' from 'sklearn.cluster' (/gladstone/engelhardt/lab/jutran/software/miniforge3/envs/AnalysisEnv/lib/python3.14/site-packages/sklearn/cluster/__init__.py)

### generate other tarHMM arrays
/gladstone/engelhardt/lab/jutran/lci/treeHMM/notebooks/formatting_caliban_input.ipynb

In [ ]:
t_cell_tracks = type_tracks_per_well["t_cell"][example_well_id]

: 